# RT-DETRv4 (RT-DETRs repo) Training Notebook

In [ ]:
!git clone https://github.com/RT-DETRs/RT-DETRv4.git
%cd RT-DETRv4
!git clone https://github.com/facebookresearch/dinov3.git
!pip install -r requirements.txt
!pip install torchmetrics

Cloning into 'RT-DETRv4'...
remote: Enumerating objects: 181, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 181 (delta 30), reused 22 (delta 22), pack-reused 127 (from 1)
Receiving objects: 100% (181/181), 5.72 MiB | 18.67 MiB/s, done.
Resolving deltas: 100% (53/53), done.
/content/RT-DETRv4
Cloning into 'dinov3'...
remote: Enumerating objects: 583, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 583 (delta 0), reused 0 (delta 0), pack-reused 579 (from 1)
Receiving objects: 100% (583/583), 12.97 MiB | 25.34 MiB/s, done.
Resolving deltas: 100% (204/204), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 26.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile
import os

dataset_path = '/content/drive/MyDrive/ai09-level1-project.zip'
output_dir = '/content/ai09-project01/'
os.makedirs(output_dir, exist_ok=True)

with zipfile.ZipFile(dataset_path) as zip_file:
    zip_file.extractall(path=output_dir)

data_root = os.path.join(output_dir, 'sprint_ai_project1_data')
train_image_dir = os.path.join(data_root, "train_images")
train_annotation_dir = os.path.join(data_root, "train_annotations")
test_image_dir = os.path.join(data_root, "test_images")

print("Train image dir:", train_image_dir)
print("Train annotation dir:", train_annotation_dir)
print("Test image dir:", test_image_dir)

Train image dir: /content/ai09-project01/sprint_ai_project1_data/train_images
Train annotation dir: /content/ai09-project01/sprint_ai_project1_data/train_annotations
Test image dir: /content/ai09-project01/sprint_ai_project1_data/test_images


In [ ]:
import os, json, shutil
from sklearn.model_selection import train_test_split

BASE_DATA = "/content/ai09-project01/sprint_ai_project1_data"
IMG_DIR = os.path.join(BASE_DATA, "train_images")
ANN_DIR = os.path.join(BASE_DATA, "train_annotations")

OUTPUT_BASE = "/content/rtdetr_dataset"
TRAIN_DIR = os.path.join(OUTPUT_BASE, "train/images")

os.makedirs(TRAIN_DIR, exist_ok=True)

In [ ]:
import pandas as pd
import cv2

def safe_get(d, keys, default=None):
    for k in keys:
        if k in d and d[k] not in [None, ""]:
            return d[k]
    return default

def get_image_files(image_dir, exts=(".png", ".jpg", ".jpeg")):
    image_files = sorted([
        f for f in os.listdir(image_dir)
        if f.lower().endswith(exts)
    ])
    print("Number of image files:", len(image_files))
    print("Sample image files:", image_files[:5])
    return image_files

def build_images_df(image_dir, image_files):
    rows = []

    for file_name in image_files:
        img_path = os.path.join(image_dir, file_name)
        img = cv2.imread(img_path)

        if img is None:
            rows.append({
                "file_name": file_name,
                "width": None,
                "height": None,
                "channel": None,
                "is_broken": True
            })
            continue

        h, w, c = img.shape
        rows.append({
            "file_name": file_name,
            "width": w,
            "height": h,
            "channel": c,
            "is_broken": False
        })

    images_df = pd.DataFrame(rows)
    images_df["image_key"] = images_df["file_name"].apply(lambda x: os.path.splitext(x)[0])
    return images_df

def parse_annotation_json(json_path, image_folder_name=None):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows = []

    if isinstance(data, dict) and "annotations" in data:
        annotations = data.get("annotations", [])
        images = data.get("images", [])
        categories = data.get("categories", [])

        image_info = images[0] if len(images) > 0 else {}
        image_id = safe_get(image_info, ["id"], default=None)
        file_name = safe_get(image_info, ["file_name"], default=None)

        category_map = {}
        for cat in categories:
            cat_id = safe_get(cat, ["id"], default=None)
            cat_name = safe_get(cat, ["name"], default=None)
            category_map[cat_id] = cat_name

        for ann in annotations:
            bbox = safe_get(ann, ["bbox"], default=None)
            category_id = safe_get(ann, ["category_id"], default=None)

            rows.append({
                "image_key": image_folder_name,
                "json_file": os.path.basename(json_path),
                "file_name_from_json": file_name,
                "image_id": image_id,
                "bbox": bbox,
                "category_id": category_id,
                "category_name": category_map.get(category_id, None),
                "dl_idx": safe_get(image_info, ["dl_idx"]),
                "dl_name": safe_get(image_info, ["dl_name"]),
                "drug_shape": safe_get(image_info, ["drug_shape"]),
                "color_class1": safe_get(image_info, ["color_class1"]),
                "color_class2": safe_get(image_info, ["color_class2"]),
                "line_front": safe_get(image_info, ["line_front"]),
                "line_back": safe_get(image_info, ["line_back"]),
                "print_front": safe_get(image_info, ["print_front"]),
                "print_back": safe_get(image_info, ["print_back"]),
                "json_path": json_path
            })
        return rows

    raise ValueError(f"Unsupported annotation format: {json_path}")

def build_annotations_df(annotation_root):
    rows = []

    for root, dirs, files in os.walk(annotation_root):
        json_files = [f for f in files if f.lower().endswith(".json")]
        for jf in json_files:
            json_path = os.path.join(root, jf)
            rel_dir = os.path.relpath(root, annotation_root)
            dir_parts = rel_dir.split(os.sep)

            json_stem = os.path.splitext(jf)[0]
            image_key = json_stem

            try:
                parsed_rows = parse_annotation_json(json_path, image_folder_name=image_key)
                for row in parsed_rows:
                    row["relative_dir"] = rel_dir
                    row["dir_depth"] = len(dir_parts)
                    rows.append(row)
            except Exception as e:
                rows.append({
                    "image_key": image_key,
                    "json_file": jf,
                    "file_name_from_json": None,
                    "image_id": None,
                    "bbox": None,
                    "category_id": None,
                    "category_name": None,
                    "dl_idx": None,
                    "dl_name": None,
                    "json_path": json_path,
                    "relative_dir": rel_dir,
                    "dir_depth": len(dir_parts),
                    "parse_error": str(e)
                })

    annotations_df = pd.DataFrame(rows)
    print("annotations_df shape:", annotations_df.shape)
    return annotations_df

In [ ]:
import yaml

def convert_loaded_annotations_to_yolo(
    images_df,
    annotations_df,
    train_image_dir,
    output_root,
    class_source="auto",
    copy_images=True,
    split_name="train"
):
    def is_valid_bbox(bbox):
        return isinstance(bbox, list) and len(bbox) == 4 and bbox[2] > 0 and bbox[3] > 0

    def xywh_to_yolo(img_w, img_h, bbox):
        x, y, w, h = bbox
        x_center = (x + w / 2.0) / img_w
        y_center = (y + h / 2.0) / img_h
        w = w / img_w
        h = h / img_h
        return x_center, y_center, w, h

    def choose_class_series(df, class_source):
        if class_source == "category_name" and "category_name" in df.columns:
            return df["category_name"]
        elif class_source == "dl_name" and "dl_name" in df.columns:
            return df["dl_name"]
        elif class_source == "category_id" and "category_id" in df.columns:
            return df["category_id"].astype(str)
        elif class_source == "dl_idx" and "dl_idx" in df.columns:
            return df["dl_idx"].astype(str)
        elif class_source == "auto":
            if "category_name" in df.columns and df["category_name"].notna().sum() > 0:
                return df["category_name"]
            elif "dl_name" in df.columns and df["dl_name"].notna().sum() > 0:
                return df["dl_name"]
            elif "category_id" in df.columns and df["category_id"].notna().sum() > 0:
                return df["category_id"].astype(str)
            elif "dl_idx" in df.columns and df["dl_idx"].notna().sum() > 0:
                return df["dl_idx"].astype(str)
            else:
                raise ValueError("No valid class column found in annotations_df.")
        else:
            raise ValueError(f"Invalid class_source: {class_source}")

    image_out_dir = os.path.join(output_root, "images", split_name)
    label_out_dir = os.path.join(output_root, "labels", split_name)
    os.makedirs(image_out_dir, exist_ok=True)
    os.makedirs(label_out_dir, exist_ok=True)

    merged_df = annotations_df.copy()
    class_series = choose_class_series(merged_df, class_source).fillna("UNKNOWN").astype(str)
    unique_classes = sorted(class_series.unique().tolist())
    class_to_id = {cls_name: idx for idx, cls_name in enumerate(unique_classes)}

    image_meta = images_df.set_index("image_key")[["file_name", "width", "height"]].to_dict("index")
    grouped = merged_df.groupby("image_key")

    valid_image_count = 0
    valid_box_count = 0

    for image_key, group in grouped:
        if image_key not in image_meta:
            continue

        file_name = image_meta[image_key]["file_name"]
        img_w = image_meta[image_key]["width"]
        img_h = image_meta[image_key]["height"]

        if pd.isna(img_w) or pd.isna(img_h):
            continue

        label_lines = []
        for _, row in group.iterrows():
            bbox = row.get("bbox", None)
            if not is_valid_bbox(bbox):
                continue

            cls_name = str(choose_class_series(group, class_source).loc[row.name]) if row.name in group.index else None
            if cls_name is None:
                continue

            cls_id = class_to_id[cls_name]
            x_c, y_c, w, h = xywh_to_yolo(img_w, img_h, bbox)
            label_lines.append(f"{cls_id} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}")
            valid_box_count += 1

        if len(label_lines) == 0:
            continue

        src_img_path = os.path.join(train_image_dir, file_name)
        dst_img_path = os.path.join(image_out_dir, file_name)
        dst_lbl_path = os.path.join(label_out_dir, f"{image_key}.txt")

        if copy_images and os.path.exists(src_img_path):
            shutil.copy2(src_img_path, dst_img_path)

        with open(dst_lbl_path, "w", encoding="utf-8") as f:
            f.write("\n".join(label_lines))

        valid_image_count += 1

    names_dict = {idx: name for name, idx in class_to_id.items()}
    yaml_path = os.path.join(output_root, "dataset.yaml")
    yaml_data = {
        "path": output_root,
        "train": f"images/{split_name}",
        "val": f"images/{split_name}",
        "names": names_dict
    }

    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(yaml_data, f, sort_keys=False, allow_unicode=True)

    print(f"YOLO dataset created at: {output_root}")
    print("Valid images:", valid_image_count)
    print("Valid boxes:", valid_box_count)
    print("Number of classes:", len(class_to_id))

    return {
        "image_dir": image_out_dir,
        "label_dir": label_out_dir,
        "yaml_path": yaml_path,
        "class_to_id": class_to_id
    }

In [ ]:
print(train_image_dir)

image_files = get_image_files(train_image_dir)
images_df = build_images_df(train_image_dir, image_files)
annotations_df = build_annotations_df(train_annotation_dir)

annotations_df = annotations_df.copy()
if "file_name_from_json" in annotations_df.columns:
    mask = annotations_df["file_name_from_json"].notna()
    annotations_df.loc[mask, "image_key"] = annotations_df.loc[mask, "file_name_from_json"].apply(
        lambda x: os.path.splitext(os.path.basename(x))[0]
    )

yolo_info = convert_loaded_annotations_to_yolo(
    images_df=images_df,
    annotations_df=annotations_df,
    train_image_dir=train_image_dir,
    output_root="/content/yolo_pill_dataset",
    class_source="auto",
    copy_images=True,
    split_name="train"
)

image_dir = yolo_info["image_dir"]
label_dir = yolo_info["label_dir"]
class_to_id = yolo_info["class_to_id"]

image_files = sorted([
    f for f in os.listdir(image_dir)
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
])

pairs = []
for img_file in image_files:
    stem = os.path.splitext(img_file)[0]
    label_file = f"{stem}.txt"
    if os.path.exists(os.path.join(label_dir, label_file)):
        pairs.append((img_file, label_file))

print("Number of image-label pairs:", len(pairs))

/content/ai09-project01/sprint_ai_project1_data/train_images
Number of image files: 232
Sample image files: ['K-001900-016548-019607-029451_0_2_0_2_70_000_200.png', 'K-001900-016548-019607-029451_0_2_0_2_75_000_200.png', 'K-001900-016548-019607-029451_0_2_0_2_90_000_200.png', 'K-001900-016548-019607-033009_0_2_0_2_70_000_200.png', 'K-001900-016548-019607-033009_0_2_0_2_75_000_200.png']
annotations_df shape: (763, 19)
YOLO dataset created at: /content/yolo_pill_dataset
Valid images: 232
Valid boxes: 763
Number of classes: 56
Number of image-label pairs: 232
Created fold yaml files:
 - /content/yolo_pill_dataset_kfold/fold_1/dataset.yaml
 - /content/yolo_pill_dataset_kfold/fold_2/dataset.yaml
 - /content/yolo_pill_dataset_kfold/fold_3/dataset.yaml
 - /content/yolo_pill_dataset_kfold/fold_4/dataset.yaml
 - /content/yolo_pill_dataset_kfold/fold_5/dataset.yaml


In [ ]:
import os, sys, json, yaml, cv2, shutil, random, math
from copy import deepcopy
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
from tqdm.notebook import tqdm

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

def find_images(d: Path):
    return sorted([p for p in d.iterdir() if p.suffix.lower() in IMG_EXTS])

def find_jsons(d: Path):
    return sorted(d.rglob("*.json"))

def load_json(p: Path):
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def clean_dir(p: Path, wipe=True):
    if wipe and p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)

def inpaint_flag(name: str):
    return cv2.INPAINT_TELEA if name.lower() == "telea" else cv2.INPAINT_NS

print("유틸 함수 로드 완료")

유틸 함수 로드 완료


In [ ]:
def load_records(image_dir: Path, ann_dir: Path):
    grouped = defaultdict(list)
    for jp in tqdm(find_jsons(ann_dir), desc="JSON 스캔"):
        data = load_json(jp)
        imgs = data.get("images", [])
        fname = imgs[0].get("file_name", "") if imgs else ""
        key = Path(fname).stem or jp.stem
        grouped[key].append((jp, data, fname))

    records = []
    for key, items in tqdm(grouped.items(), desc="레코드 생성"):
        rep_jp, rep_data, rep_fname = items[0]

        img_path = None
        for fname_try in [rep_fname] + [f"{key}{ext}" for ext in IMG_EXTS]:
            p = image_dir / Path(fname_try).name
            if p.exists():
                img_path = p
                break

        anns, ann_dicts = [], []
        for jp, data, _ in items:
            for ann in data.get("annotations", []):
                bbox = ann.get("bbox")
                if isinstance(bbox, list) and len(bbox) == 4 and bbox[2] > 0 and bbox[3] > 0:
                    a = deepcopy(ann)
                    a["_src_json"] = str(jp)
                    a["_src_rel"]  = str(jp.relative_to(ann_dir))
                    ann_dicts.append(a)
                    anns.append({"bbox": bbox, "category_id": ann.get("category_id")})

        merged = deepcopy(rep_data)
        merged["annotations"] = ann_dicts
        if not merged.get("images"):
            merged["images"] = [{"file_name": rep_fname}]

        records.append({
            "image_key":       key,
            "image_path":      img_path,
            "image_file_name": Path(rep_fname).name or f"{key}.jpg",
            "json_paths":      [str(jp) for jp, _, _ in items],
            "annotations":     anns,
            "data":            merged,
        })
    return records

records = load_records(Path(train_image_dir), Path(train_annotation_dir))
matched   = [r for r in records if r["image_path"] is not None]
unmatched = [r for r in records if r["image_path"] is None]
ann_counts = [len(r["annotations"]) for r in records]

print(f"총 레코드: {len(records)}개  (이미지 매칭 성공: {len(matched)}, 실패: {len(unmatched)})")
print(f"annotation 수 — min: {min(ann_counts)}, max: {max(ann_counts)}, mean: {sum(ann_counts)/len(ann_counts):.2f}")
if unmatched:
    print("매칭 실패 예시:", [r["image_key"] for r in unmatched[:5]])

JSON 스캔:   0%|          | 0/763 [00:00<?, ?it/s]

레코드 생성:   0%|          | 0/232 [00:00<?, ?it/s]

총 레코드: 232개  (이미지 매칭 성공: 232, 실패: 0)
annotation 수 — min: 2, max: 4, mean: 3.29


In [ ]:
all_cats = set()
for r in records:
    for ann in r["annotations"]:
        all_cats.add(ann["category_id"])

sorted_cats = sorted(list(all_cats))
cat2id = {cat: idx for idx, cat in enumerate(sorted_cats)}
id2cat = {idx: cat for cat, idx in cat2id.items()}
num_classes = len(cat2id)
print("num classes:", num_classes)

train_records = records

for r in records:
    for ann in r["annotations"]:
        if ann["category_id"] in cat2id:
            ann["category_id"] = cat2id[ann["category_id"]]

num classes: 56


In [ ]:
# =========================
# 🔥 Inpainting 기반 Augmentation (붙여넣기용)
# =========================

import cv2
import numpy as np
import random
import os
from tqdm import tqdm

# -------------------------
# 1. 객체 제거 (inpainting)
# -------------------------
def remove_objects_with_inpainting(image, annotations):
    mask = np.zeros(image.shape[:2], dtype=np.uint8)

    for ann in annotations:
        x, y, w, h = map(int, ann["bbox"])
        mask[y:y+h, x:x+w] = 255

    inpainted = cv2.inpaint(image, mask, 3, cv2.INPAINT_TELEA)
    return inpainted


# -------------------------
# 2. 객체 crop 추출
# -------------------------
def extract_objects(image, annotations):
    crops = []
    for ann in annotations:
        x, y, w, h = map(int, ann["bbox"])
        crop = image[y:y+h, x:x+w]

        if crop.size == 0:
            continue

        crops.append({
            "image": crop,
            "category_id": ann["category_id"]
        })

    return crops


# -------------------------
# 3. bbox 겹침 방지 함수
# -------------------------
def is_overlap(box1, box2):
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2

    if (x1 + w1 < x2) or (x2 + w2 < x1):
        return False
    if (y1 + h1 < y2) or (y2 + h2 < y1):
        return False
    return True


# -------------------------
# 4. 객체 붙이기
# -------------------------
def paste_objects(image, objects, max_objects=4):
    h, w = image.shape[:2]
    new_anns = []

    for obj in objects:
        crop = obj["image"]
        cid = obj["category_id"]

        oh, ow = crop.shape[:2]

        if oh >= h or ow >= w:
            continue

        for _ in range(10):  # 최대 10번 시도
            x = random.randint(0, w - ow)
            y = random.randint(0, h - oh)

            new_box = [x, y, ow, oh]

            # 기존 bbox와 겹치는지 확인
            if any(is_overlap(new_box, ann["bbox"]) for ann in new_anns):
                continue

            image[y:y+oh, x:x+ow] = crop

            new_anns.append({
                "bbox": new_box,
                "category_id": cid
            })
            break

    return image, new_anns


# -------------------------
# 5. 클래스 분포 계산 (불균형 대응)
# -------------------------
from collections import Counter

def get_class_distribution(records):
    counter = Counter()
    for r in records:
        for ann in r["annotations"]:
            counter[ann["category_id"]] += 1
    return counter


# -------------------------
# 6. Augmentation 실행
# -------------------------
def generate_augmented_data(train_records, image_dir, save_dir, repeat=2):
    os.makedirs(save_dir, exist_ok=True)

    class_dist = get_class_distribution(train_records)
    max_count = max(class_dist.values())

    augmented_records = []

    for r in tqdm(train_records, desc="Augmenting"):
        if r["image_path"] is None:
            continue

        img = cv2.imread(str(r["image_path"]))
        if img is None:
            continue

        # 객체 crop
        crops = extract_objects(img, r["annotations"])
        if len(crops) == 0:
            continue

        # 클래스 기반 repeat 조절
        avg_class = int(np.mean([c["category_id"] for c in crops]))
        class_count = class_dist.get(avg_class, 1)

        imbalance_ratio = max_count / class_count
        dynamic_repeat = int(min(5, max(1, imbalance_ratio)))

        for i in range(repeat * dynamic_repeat):

            # 1. 객체 제거
            clean_img = remove_objects_with_inpainting(img, r["annotations"])

            # 2. 랜덤 선택
            sampled = random.sample(crops, min(len(crops), random.randint(1, 4)))

            # 3. 붙이기
            new_img, new_anns = paste_objects(clean_img.copy(), sampled)

            if len(new_anns) == 0:
                continue

            new_file_name = f"aug_{i}_{r['image_file_name']}"
            save_path = os.path.join(save_dir, new_file_name)

            cv2.imwrite(save_path, new_img)

            new_record = {
                "image_file_name": new_file_name,
                "image_path": save_path,
                "annotations": new_anns,
                "data": r.get("data", {})
            }

            augmented_records.append(new_record)

    return augmented_records

AUG_IMAGE_DIR = os.path.join(TRAIN_DIR, "augmented")
augmented_records = generate_augmented_data(
    train_records=train_records,
    image_dir=TRAIN_DIR,
    save_dir=AUG_IMAGE_DIR,
    repeat=2  # 기본 2배
)

train_records_extended = train_records + augmented_records

# -------------------------
# 기존 데이터 + 증강 데이터 합치기
# -------------------------
print(f"원본: {len(train_records)}")
print(f"증강: {len(augmented_records)}")
print(f"총합: {len(train_records_extended)}")

Augmenting: 100%|██████████| 185/185 [09:00<00:00,  2.92s/it]

원본: 185
증강: 1850
총합: 2035


In [ ]:
def convert_to_coco(records, save_path):
    images, annotations = [], []
    ann_id = 1

    for img_id, r in enumerate(records):
        images.append({
            "id": img_id,
            "file_name": r["image_file_name"],
            "width": r["data"]["images"][0]["width"],
            "height": r["data"]["images"][0]["height"]
        })

        for ann in r["annotations"]:
            x, y, w, h = ann["bbox"]
            cid = ann["category_id"]

            if cid < 0 or cid >= len(cat2id):
              continue
            if w <= 0 or h <= 0:
              continue

            annotations.append({
                "id": ann_id,
                "image_id": img_id,
                "category_id": cid,
                "bbox": [x, y, w, h],
                "area": w*h,
                "iscrowd": 0
            })
            ann_id += 1

    coco = {
        "images": images,
        "annotations": annotations,
        "categories": [{"id": i, "name": c} for c,i in cat2id.items()]
    }

    with open(save_path, "w") as f:
        json.dump(coco, f)

In [ ]:
def copy_images(records, dest):
    for r in records:
        src = os.path.join(IMG_DIR, r["image_file_name"])
        dst = os.path.join(dest, r["image_file_name"])
        if os.path.exists(src):
            shutil.copy(src, dst)

copy_images(train_records, TRAIN_DIR)

In [ ]:
train_json = os.path.join(OUTPUT_BASE, "train.json")

convert_to_coco(train_records_extended, train_json)

print("COCO ready")

COCO ready


## Config 수정 (RT-DETRv4는 configs 아래 yaml 수정 필요)

In [ ]:
config_path = "configs/dataset/custom.yaml"

config_text = f'''
train: {TRAIN_DIR}

train_ann: {train_json}

num_classes: {num_classes}
'''

with open(config_path, "w") as f:
    f.write(config_text)

print("config saved:", config_path)

config saved: configs/dataset/custom.yaml


In [ ]:
import json

with open("/content/rtdetr_dataset/train.json") as f:
    data = json.load(f)

print(len(data["images"]))
print(len(data["annotations"]))
print(data["annotations"][:3])

47
154
[{'id': 1, 'image_id': 0, 'category_id': 3, 'bbox': [610, 791, 272, 211], 'area': 57392, 'iscrowd': 0}, {'id': 2, 'image_id': 0, 'category_id': 33, 'bbox': [51, 148, 449, 425], 'area': 190825, 'iscrowd': 0}, {'id': 3, 'image_id': 0, 'category_id': 36, 'bbox': [123, 766, 317, 277], 'area': 87809, 'iscrowd': 0}]
2035
4822
[{'id': 1, 'image_id': 0, 'category_id': 12, 'bbox': [190, 299, 176, 251], 'area': 44176, 'iscrowd': 0}, {'id': 2, 'image_id': 0, 'category_id': 53, 'bbox': [541, 302, 211, 238], 'area': 50218, 'iscrowd': 0}, {'id': 3, 'image_id': 1, 'category_id': 4, 'bbox': [149, 188, 201, 199], 'area': 39999, 'iscrowd': 0}]


In [ ]:
import os
import shutil

src_dir = "/content/rtdetr_dataset/train/images/augmented"
dst_dir = "/content/rtdetr_dataset/train/images"

for fname in os.listdir(src_dir):
    src_path = os.path.join(src_dir, fname)
    dst_path = os.path.join(dst_dir, fname)

    if os.path.isfile(src_path):
        shutil.move(src_path, dst_path)

In [ ]:
!rm -rf /content/rtdetr_dataset/train/images/augmented

In [ ]:
!python train.py --config configs/rtv4/custom_rtdetrv4_config_aug.yml --output-dir rt_detr_work

2026-03-31 06:01:39.183568: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774936899.205094    5399 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774936899.212320    5399 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774936899.230493    5399 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774936899.230522    5399 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774936899.230526    5399 computation_placer.cc:177] computation placer alr